In [1]:
## java runtime is no more available in colab. We'll have to install it manually.
!apt-get install openjdk-21-jdk-headless -qq > /dev/null

In [2]:
!pip install findspark

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [4]:
# Set SPARK_HOME to the PySpark installation path
# This is where the 'bin/spark-submit' file is located
import pyspark
import findspark
os.environ["SPARK_HOME"] = pyspark.__file__.replace('__init__.py', '')

# Initialize findspark to configure Spark environment paths
findspark.init()

In [5]:
# Verification step
!ls $SPARK_HOME/bin/spark-submit

/usr/local/lib/python3.13/dist-packages/pyspark//bin/spark-submit


In [6]:
from pyspark.sql import SparkSession

#Create SparkSession
spark = SparkSession.builder.appName('products').master('local[*]').config('spark.driver.memory', '4g').getOrCreate()
sc=spark.sparkContext
print(spark.version)

4.0.5


In [8]:
pr_list=["Iphone","Samsung","Realme","Airpods","Charger"]

In [9]:
pr_rdd=sc.parallelize(pr_list)

In [10]:
type(pr_rdd)

pyspark.core.rdd.RDD

In [11]:
pr_list

['Iphone', 'Samsung', 'Realme', 'Airpods', 'Charger']

In [12]:
pr_rdd

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:298

In [13]:
#Retrieve all the element  in rdd
pr_rdd.collect()

['Iphone', 'Samsung', 'Realme', 'Airpods', 'Charger']

In [14]:
#Get the output Realme
pr_rdd.collect()[2]

'Realme'

In [15]:
pr_rdd.collect()[-1].upper()

'CHARGER'

In [16]:
pr_rdd.collect()[0].lower()

'iphone'

In [17]:
pr_rdd.collect()[1:3]

['Samsung', 'Realme']

In [18]:
pr_rdd.collect()[::2]

['Iphone', 'Realme', 'Charger']

In [19]:
pr_rdd.collect()

['Iphone', 'Samsung', 'Realme', 'Airpods', 'Charger']

In [20]:
from pyspark import SparkConf,SparkContext
conf = SparkConf().set('spark.ui.port','4051').setAppName('test1').setMaster('local[2]')
sc = SparkContext.getOrCreate(conf=conf)
#rdd1 = sc.parallelize([2,3,4,5,6,7,11])

#rdd1 = sc.parallelize([2,3,4,5,6,7,11],8)#(no of element,how many partition)
#rdd1.glom().collect()#How the data is divided into partition

In [21]:
n=[10,20,30,40,50,60,70]
#Convert list data into rdd
rdd1=sc.parallelize(n,10)#How many partition-10
rdd1.collect()

[10, 20, 30, 40, 50, 60, 70]

In [22]:
#How the partition is dividing
rdd1.glom().collect()

[[], [10], [20], [], [30], [40], [], [50], [60], [70]]

In [23]:
n=[10,20,30,40,50,60,70]
#Convert list data into rdd
rdd2=sc.parallelize(n,3)#How many partition-10
rdd2.glom().collect()

[[10, 20], [30, 40], [50, 60, 70]]

In [24]:
rdd1.stats()

(count: 7, mean: 40.0, stdev: 20.0, max: 70.0, min: 10.0)

In [25]:
rdd1.getNumPartitions()

10

In [26]:
rdd2.getNumPartitions()

3

In [31]:
rdd1.coalesce(15).glom().collect() #Reduce the partition
#increase it will not when we convert the data into rdd we mention 10 partition
#it will not increase beyond the partition

[[], [10], [20], [], [30], [40], [], [50], [60], [70]]

In [29]:
rdd1.repartition(12).glom().collect()

[[60], [], [30, 40], [], [10], [20, 50], [], [], [], [70], [], []]

In [33]:
pr_rdd.collect()

['Iphone', 'Samsung', 'Realme', 'Airpods', 'Charger']

In [35]:
pr_rdd.take(3)

['Iphone', 'Samsung', 'Realme']

In [ ]:
transform = lambda i:i.upper()
list(map(transform,pr_rdd))

In [42]:
def upper(n):
  return n.upper()
upper("Python")
z=map(upper,pr_list)
list(z)

['IPHONE', 'SAMSUNG', 'REALME', 'AIRPODS', 'CHARGER']

In [41]:
pr_list

['Iphone', 'Samsung', 'Realme', 'Airpods', 'Charger']

In [43]:
#lambda variable:expression
y=list(map(lambda n:n.upper(),pr_list))
y

['IPHONE', 'SAMSUNG', 'REALME', 'AIRPODS', 'CHARGER']

In [44]:
pr_rdd.collect()

['Iphone', 'Samsung', 'Realme', 'Airpods', 'Charger']

In [46]:
transform = lambda i:i.upper()
pr_title_rdd = pr_rdd.map(transform).collect()
pr_title_rdd
#movie_title_rdd.collect()

['IPHONE', 'SAMSUNG', 'REALME', 'AIRPODS', 'CHARGER']

In [47]:
transform=lambda i:i[0]=="C"
transform("Charger")

True

In [48]:
pr_title_rdd = pr_rdd.filter(transform).collect()
pr_title_rdd

['Charger']

In [49]:
# Actions
# Count the number of elements in the RDD
num_pr = pr_rdd.count()
print(f"Number of Product: {num_pr}")


Number of Product: 5


In [50]:
# Find the first element in the RDD
first_pr = pr_rdd.first()
print(f"First Product: {first_pr}")


First Product: Iphone


In [56]:
concatenated_pr = pr_rdd.reduce(lambda x, y: x +','+ y)
print(f"Concatenated movie titles: {concatenated_pr}")

Concatenated movie titles: Iphone,Samsung,Realme,Airpods,Charger


In [51]:
from functools import reduce
x=[10,20,30,40]
y=reduce(lambda a,b:(a+b),x)
y

100

In [57]:
pr_rdd.collect()

['Iphone', 'Samsung', 'Realme', 'Airpods', 'Charger']

In [60]:
# Transformations
# Map each pr title to its length
pr_lengths = pr_rdd.map(lambda x: len(x))
print("Product lengths:", pr_lengths.collect())


Product lengths: [6, 7, 6, 7, 7]


In [63]:
#Filter the product which is equal to 6
pr_lengths_filter = pr_rdd.filter(lambda x: len(x)==6)
print("Product lengths:", pr_lengths_filter.collect())


Product lengths: ['Iphone', 'Realme']


In [64]:
#Greater than 6 product length
pr_lengths_filter = pr_rdd.filter(lambda x: len(x)>6)
print("Product lengths:", pr_lengths_filter.collect())

Product lengths: ['Samsung', 'Airpods', 'Charger']


In [65]:
# FlatMap: Explode a list into individual elements
pr_chars_rdd = pr_rdd.flatMap(lambda x : list(x))
print("Characters:",pr_chars_rdd.collect())


Characters: ['I', 'p', 'h', 'o', 'n', 'e', 'S', 'a', 'm', 's', 'u', 'n', 'g', 'R', 'e', 'a', 'l', 'm', 'e', 'A', 'i', 'r', 'p', 'o', 'd', 's', 'C', 'h', 'a', 'r', 'g', 'e', 'r']


In [66]:
pr_chars_rdd.distinct().collect()

['I',
 'p',
 'o',
 'S',
 'g',
 'R',
 'l',
 'A',
 'd',
 'h',
 'n',
 'e',
 'a',
 'm',
 's',
 'u',
 'i',
 'r',
 'C']

In [67]:
# GroupByKey: Group the elements based on a key
# We need key-value pairs for this transformation.
# Let's create a new RDD
pr_with_length = pr_rdd.map(lambda x: (len(x),x))
print(pr_with_length.collect())


[(6, 'Iphone'), (7, 'Samsung'), (6, 'Realme'), (7, 'Airpods'), (7, 'Charger')]


In [69]:
grouped_pr = pr_with_length.groupByKey().mapValues(list).collect()
print("Product grouped by length:", grouped_pr)

Product grouped by length: [(6, ['Iphone', 'Realme']), (7, ['Samsung', 'Airpods', 'Charger'])]


In [70]:

# SortByKey
sorted_pr = pr_with_length.sortByKey().collect()
print("Product sorted by Title:", sorted_pr)

Product sorted by Title: [(6, 'Iphone'), (6, 'Realme'), (7, 'Samsung'), (7, 'Airpods'), (7, 'Charger')]


In [71]:
rdd1 = sc.parallelize([2,3,4,5,6,7,11])

In [72]:
rdd1.glom().collect()

[[2, 3, 4], [5, 6, 7, 11]]

In [73]:
rdd2 = rdd1.filter(lambda x: x %2==0)
rdd2.collect()

[2, 4, 6]

In [74]:
rdd2 = rdd1.filter(lambda x: x %2!=0)
rdd2.collect()

[3, 5, 7, 11]

In [75]:
rdd2 = rdd1.map(lambda x: x *10)
rdd2.collect()

[20, 30, 40, 50, 60, 70, 110]

In [76]:
rdd3 = sc.parallelize([[2,3,4],[5,6,7],[11,12,13]])
rdd2 = rdd3.map(lambda iter: [x * 2 for x in iter])
rdd2.collect()

[[4, 6, 8], [10, 12, 14], [22, 24, 26]]

In [79]:
x=[2,3,4,5,6,7,11,12,13]
for i in x:
  print(i*2,end=" ")

4 6 8 10 12 14 22 24 26 

In [81]:
rdd3 = sc.parallelize([[2,3,4],[5,6,7],[11,12,13]])
rdd2 = rdd3.mapPartitions(lambda iter: [x * 3 for x in iter])
rdd2.collect()


[[2, 3, 4, 2, 3, 4, 2, 3, 4],
 [5, 6, 7, 5, 6, 7, 5, 6, 7],
 [11, 12, 13, 11, 12, 13, 11, 12, 13]]

In [82]:
rdd1.collect(),rdd3.collect()

([2, 3, 4, 5, 6, 7, 11], [[2, 3, 4], [5, 6, 7], [11, 12, 13]])

In [83]:
rdd2=rdd1.union(rdd3)

In [84]:
rdd2.collect()

[2, 3, 4, 5, 6, 7, 11, [2, 3, 4], [5, 6, 7], [11, 12, 13]]

In [85]:
rdd3 = sc.parallelize([7,11,12,13])
rdd2 = rdd1.intersection(rdd3)
rdd2.collect()

[7, 11]